In [ ]:
import csv, re, difflib
import numpy as np
from scipy.stats import spearmanr

GOLD_FILE       = "/content/drive/MyDrive/Paper5LLMasEval/GoldDataset.csv"
CAND_FILE       = "/content/drive/MyDrive/Paper5LLMasEval/CandidatesDataset.csv"
OUT_FILE        = "/content/drive/MyDrive/Paper5LLMasEval/BaselineScores.csv"
FUZZY_THRESHOLD = 0.80        # fixed a priori, not tuned

# ---------------- name / type normalisation ----------------------
def norm(name):
    return re.sub(r'[^a-z0-9]', '', name.lower())

def singular(n):
    return n[:-1] if len(n) > 3 and n.endswith('s') and not n.endswith('ss') else n

TYPE_MAP = {'integer': 'int', 'long': 'int', 'short': 'int', 'int': 'int',
            'double': 'float', 'float': 'float', 'real': 'float', 'decimal': 'float',
            'number': 'float', 'bool': 'bool', 'boolean': 'bool',
            'str': 'string', 'string': 'string', 'text': 'string', 'char': 'string',
            'date': 'date', 'datetime': 'date', 'localdate': 'date',
            'localdatetime': 'date', 'timestamp': 'date', 'time': 'time'}
def norm_type(t):
    t = norm(t or '')
    return TYPE_MAP.get(t, t)

def mult_kind(m):
    """Compare multiplicities by meaning: one vs. many (upper bound)."""
    if m is None:
        return None
    m = m.strip().lower().replace(' ', '')
    if m == '':
        return None
    upper = m.split('..')[-1]
    return 'many' if upper in ('*', 'n', 'm', 'many') or (upper.isdigit() and int(upper) > 1) else 'one'

# ---------------- PlantUML parsing --------------------------------
DECL = re.compile(r'^\s*(abstract\s+class|abstract|class|interface|enum|entity)\s+'
                  r'"?([\w .]+?)"?(?:\s+as\s+(\w+))?\s*(<<[^>]*>>)?\s*(\{)?\s*$', re.I)
REL = re.compile(r'^\s*"?([\w.]+)"?\s*(?:"([^"]*)")?\s*'
                 r'(<\|?|\*|o|#|x|\+|\^)?([-.]+(?:\w+[-.]+)?)(\|?>|\*|o|#|x|\+|\^)?'
                 r'\s*(?:"([^"]*)")?\s*"?([\w.]+)"?\s*(?::.*)?$')
ATTR_OUTSIDE = re.compile(r'^\s*"?(\w+)"?\s*:\s*(.+)$')

def parse_attr(line):
    line = line.strip()
    if not line or '(' in line or line.startswith(("'", '..', '--', '==', '__')):
        return None
    line = re.sub(r'^[+\-#~]\s*', '', line)
    line = re.sub(r'\{(static|abstract|field|classifier)\}\s*', '', line, flags=re.I)
    if ':' in line:
        name, typ = line.split(':', 1)
    else:
        parts = line.split()
        if len(parts) >= 2:
            typ, name = ' '.join(parts[:-1]), parts[-1]
        else:
            name, typ = parts[0], ''
    name = re.sub(r'[^\w]', '', name)
    typ = re.sub(r'\s*=.*$', '', typ).strip()
    return (name, typ) if name else None

def rel_type(left, line_, right):
    left, right = left or '', right or ''
    dashed = '.' in line_
    if '|' in left + right or '^' in left + right:
        return 'realization' if dashed else 'inheritance'
    if '*' in left + right:
        return 'composition'
    if 'o' in left + right:
        return 'aggregation'
    return 'dependency' if dashed else 'association'

def parse_plantuml(text):
    classes, attrs, rels, alias = {}, [], [], {}
    current, is_enum = None, False
    for raw in (text or '').splitlines():
        line = raw.strip()
        if not line or line.startswith(("'", '@', 'skinparam', 'hide', 'show', 'note', 'title',
                                         'package', 'namespace', 'left to right', 'top to bottom')):
            if current is not None and line == '}':
                current = None
            continue
        if current is not None:
            if line.startswith('}'):
                current = None
            elif not is_enum:
                a = parse_attr(line)
                if a:
                    attrs.append((current, a[0], a[1]))
            continue
        m = DECL.match(line)
        if m:
            kw, name, al, stereo, brace = m.groups()
            kw = kw.lower()
            st = ('abstract' if 'abstract' in kw else
                  'interface' if kw == 'interface' else
                  'enum' if kw == 'enum' else 'none')
            if stereo and st == 'none':
                s = stereo.lower()
                st = 'interface' if 'interface' in s else 'enum' if 'enum' in s else \
                     'abstract' if 'abstract' in s else 'none'
            name = name.strip()
            classes[name] = st
            if al:
                alias[al] = name
            if brace:
                current, is_enum = name, (st == 'enum')
            continue
        m = REL.match(line)
        if m and (m.group(3) or m.group(5) or len(m.group(4)) >= 2):
            a, ma, lh, ln, rh, mb, b = m.groups()
            a, b = alias.get(a, a), alias.get(b, b)
            for c in (a, b):
                classes.setdefault(c, 'none')
            rels.append((a, b, rel_type(lh, ln, rh), mult_kind(ma), mult_kind(mb)))
            continue
        m = ATTR_OUTSIDE.match(line)
        if m and m.group(1) in classes:
            a = parse_attr(m.group(2))
            if a:
                attrs.append((m.group(1), a[0], a[1]))
    return classes, attrs, rels

# ---------------- matching ----------------------------------------
def names_match(x, y, fuzzy):
    a, b = singular(norm(x)), singular(norm(y))
    if a == b:
        return True
    return fuzzy and difflib.SequenceMatcher(None, a, b).ratio() >= FUZZY_THRESHOLD

def one_to_one(cands, golds, key_c, key_g, fuzzy):
    """Greedy one-to-one matching, exact matches first."""
    pairs, used_c, used_g = [], set(), set()
    for pass_fuzzy in ([False, True] if fuzzy else [False]):
        for i, c in enumerate(cands):
            if i in used_c:
                continue
            for j, g in enumerate(golds):
                if j in used_g:
                    continue
                if names_match(key_c(c), key_g(g), pass_fuzzy):
                    pairs.append((i, j)); used_c.add(i); used_g.add(j)
                    break
    return pairs

def ratio(num, den):
    return 1.0 if den == 0 else num / den

def score(gold_txt, cand_txt, fuzzy):
    gC, gA, gR = parse_plantuml(gold_txt)
    cC, cA, cR = parse_plantuml(cand_txt)
    gcl, ccl = list(gC), list(cC)

    # classes
    cp = one_to_one(ccl, gcl, lambda x: x, lambda x: x, fuzzy)
    cmap = {ccl[i]: gcl[j] for i, j in cp}
    cls_cmp = ratio(len(cp), len(gcl))
    cls_crt = ratio(sum(cC[ccl[i]] == gC[gcl[j]] for i, j in cp), len(ccl))

    # attributes (only within matched classes)
    matched_attr, correct_attr = 0, 0
    for cc, gc in cmap.items():
        ca = [a for a in cA if a[0] == cc]
        ga = [a for a in gA if a[0] == gc]
        ap = one_to_one(ca, ga, lambda x: x[1], lambda x: x[1], fuzzy)
        matched_attr += len(ap)
        correct_attr += sum(norm_type(ca[i][2]) == norm_type(ga[j][2]) for i, j in ap)
    atr_cmp = ratio(matched_attr, len(gA))
    atr_crt = ratio(correct_attr, len(cA))

    # relationships: same unordered pair of matched classes
    used, m_rel, c_rel = set(), 0, 0
    for (a, b, t, ma, mb) in cR:
        ga, gb = cmap.get(a), cmap.get(b)
        if ga is None or gb is None:
            continue
        for k, (x, y, gt, gma, gmb) in enumerate(gR):
            if k in used:
                continue
            if {x, y} == {ga, gb}:
                used.add(k); m_rel += 1
                if (x, y) != (ga, gb):              # same pair, reversed
                    gma, gmb = gmb, gma
                ok = t == gt
                if ok and t in ('association', 'aggregation', 'composition'):
                    ok = all(p is None or q is None or p == q for p, q in ((ma, gma), (mb, gmb)))
                c_rel += ok
                break
    rel_cmp = ratio(m_rel, len(gR))
    rel_crt = ratio(c_rel, len(cR))
    return [cls_cmp, cls_crt, atr_cmp, atr_crt, rel_cmp, rel_crt]

# ---------------- run ---------------------------------------------
def read_csv(path):
    for enc in ('utf-8-sig', 'cp1252', 'latin-1'):
        try:
            with open(path, newline='', encoding=enc) as f:
                return list(csv.DictReader(f))
        except UnicodeDecodeError:
            continue

CELLS = ['class_completeness', 'class_correctness', 'attr_completeness',
         'attr_correctness', 'rel_completeness', 'rel_correctness']

if __name__ == '__main__':
    gold = {r['id']: r for r in read_csv(GOLD_FILE)}
    cands = read_csv(CAND_FILE)
    rows, H, E, F = [], [], [], []
    for r in cands:
        g = gold[r['id']]['solution']
        h = [float(r[f'score{i}']) for i in range(1, 7)]
        e, fz = score(g, r['student_answer'], False), score(g, r['student_answer'], True)
        H.append(h); E.append(e); F.append(fz)
        row = {'candidate_id': r['id']}
        for k, c in enumerate(CELLS):
            row[f'human_{c}'] = h[k]; row[f'exact_{c}'] = round(e[k], 3); row[f'fuzzy_{c}'] = round(fz[k], 3)
        rows.append(row)
    with open(OUT_FILE, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)

    H, E, F = map(np.array, (H, E, F))
    print(f"{'cell':20s} {'exact rho':>9s} {'MAE':>6s} | {'fuzzy rho':>9s} {'MAE':>6s}")
    for k, c in enumerate(CELLS):
        out = []
        for X in (E, F):
            rho = spearmanr(H[:, k], X[:, k]).correlation if np.std(X[:, k]) > 0 and np.std(H[:, k]) > 0 else float('nan')
            out += [rho, np.abs(H[:, k] - X[:, k]).mean()]
        print(f"{c:20s} {out[0]:9.3f} {out[1]:6.3f} | {out[2]:9.3f} {out[3]:6.3f}")
    print(f"\nSaved: {OUT_FILE}")

cell                 exact rho    MAE | fuzzy rho    MAE
class_completeness       0.460  0.172 |     0.439  0.147
class_correctness          nan  0.309 |       nan  0.280
attr_completeness        0.831  0.370 |     0.806  0.328
attr_correctness         0.484  0.449 |     0.485  0.419
rel_completeness         0.559  0.246 |     0.528  0.214
rel_correctness          0.309  0.550 |     0.158  0.518

Saved: /content/drive/MyDrive/Paper5LLMasEval/BaselineScores.csv
